In [1]:
from dotenv import load_dotenv
import nest_asyncio

nest_asyncio.apply()
load_dotenv()


True

# Generate Synthetic Test Dataset

In [2]:
from utils.utils import load_from_pickle

all_documents = load_from_pickle('all_documents')

In [5]:
from ragas.testset.generator import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context
from utils.llm import get_llm, get_embedding_model

# generator with openai models
generator_llm = get_llm("gpt-4o-mini")
critic_llm = get_llm("gpt-4o-mini")
embeddings = get_embedding_model("text-embedding-3-small")

generator = TestsetGenerator.from_langchain(
    generator_llm,
    critic_llm,
    embeddings
)

# generate testset
testset = generator.generate_with_langchain_docs(all_documents, test_size=100, distributions={simple: 0.5, reasoning: 0.25, multi_context: 0.25})

Generating: 100%|██████████| 100/100 [34:24<00:00, 20.65s/it] 


In [6]:
from utils.utils import save_to_pickle

save_to_pickle(testset, 'testset')

Failed to batch ingest runs: LangSmithRateLimitError('Rate limit exceeded for https://api.smith.langchain.com/runs/batch. HTTPError(\'429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/batch\', \'{"detail":"Monthly unique traces usage limit exceeded"}\')')
Failed to batch ingest runs: LangSmithRateLimitError('Rate limit exceeded for https://api.smith.langchain.com/runs/batch. HTTPError(\'429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/batch\', \'{"detail":"Monthly unique traces usage limit exceeded"}\')')
Failed to batch ingest runs: LangSmithRateLimitError('Rate limit exceeded for https://api.smith.langchain.com/runs/batch. HTTPError(\'429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/batch\', \'{"detail":"Monthly unique traces usage limit exceeded"}\')')


# Evaluation

In [2]:
import random
import string

def generate_id():
    return ''.join(random.choices(string.ascii_letters + string.digits, k=8)).lower()

In [3]:
from utils.utils import load_from_pickle

testset = load_from_pickle('testset')
dataset = testset.to_pandas()

c:\Users\arkan\Downloads\University\TA\ta-rag-demo\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
dataset = dataset[:30]

In [5]:
dataset

,question,contexts,ground_truth,evolution_type,metadata,episode_done
0,What challenges does the concept of kearifan l...,[DOC NAME: Undang-undang Nomor 15 Tahun 2023 T...,The concept of kearifan lokal in Bali faces ch...,simple,[{'title': 'Undang-undang Nomor 15 Tahun 2023 ...,True
1,What specific responsibilities does Lembaga Pe...,[DOC NAME: Undang-undang Nomor 4 Tahun 2023 Te...,Lembaga Penjamin Simpanan (LPS) has specific r...,simple,[{'title': 'Undang-undang Nomor 4 Tahun 2023 T...,True
2,What options do residents have to obtain addit...,[DOC NAME: Undang-undang Nomor 17 Tahun 2023 T...,Residents can obtain additional benefits throu...,simple,[{'title': 'Undang-undang Nomor 17 Tahun 2023 ...,True
3,What are the penalties and additional actions ...,[DOC NAME: Undang-undang Nomor 12 Tahun 2022 T...,According to Undang-undang Nomor 12 Tahun 2022...,simple,[{'title': 'Undang-undang Nomor 12 Tahun 2022 ...,True
4,What types of recognition can be awarded to in...,[DOC NAME: Undang-undang Nomor 11 Tahun 2022 T...,Individual athletes (Olahragawan) can be award...,simple,[{'title': 'Undang-undang Nomor 11 Tahun 2022 ...,True
5,What is the role of Joko Widodo in the enactme...,[DOC NAME: Undang-undang Nomor 13 Tahun 2022 T...,Joko Widodo is the President of the Republic o...,simple,[{'title': 'Undang-undang Nomor 13 Tahun 2022 ...,True
6,What are the general eligibility requirements ...,[DOC NAME: Undang-undang Nomor 4 Tahun 2023 Te...,The general eligibility requirements for parti...,simple,[{'title': 'Undang-undang Nomor 4 Tahun 2023 T...,True
7,What is the purpose of the dana jaminan penuga...,[DOC NAME: Undang-undang Nomor 19 Tahun 2023 T...,Dana dalam rekening dana jaminan penugasan pem...,simple,[{'title': 'Undang-undang Nomor 19 Tahun 2023 ...,True
8,What is the role of Pemerintah Pusat in the go...,[DOC NAME: Undang-undang Nomor 6 Tahun 2023 Te...,"Pemerintah Pusat, which is the President of th...",simple,[{'title': 'Undang-undang Nomor 6 Tahun 2023 T...,True
9,What factors determine the allocation of budge...,[DOC NAME: Undang-undang Nomor 1 Tahun 2022 Te...,The allocation of budget for each regional app...,simple,[{'title': 'Undang-undang Nomor 1 Tahun 2022 T...,True


In [6]:
# from utils.rag.basic_agent import caller_with_sources
from utils.rag.open_source.basic_agent import caller_with_sources

questions = []
ground_truths = []
answers = []
contexts = []

for i, row in dataset.iterrows():
    print(f"Processing {i}/{len(dataset)}")

    question = row["question"]
    questions.append(question)
    ground_truths.append(row["ground_truth"])

    answer, sources = caller_with_sources(question, generate_id())
    answers.append(answer)

    content = []
    for source in sources:
        content.append(source.page_content)

    contexts.append(content)

Processing 0/30


c:\Users\arkan\Downloads\University\TA\ta-rag-demo\env\lib\site-packages\langchain_core\_api\deprecation.py:139: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(
c:\Users\arkan\Downloads\University\TA\ta-rag-demo\env\lib\site-packages\langchain_core\_api\deprecation.py:139: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(
Token indices sequence length is longer than the specified maximum sequence length for this model (1154 > 1024). Running this sequence through the model will result in indexing errors
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/b

Processing 1/30


INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-m3


Processing 2/30


INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-m3


... (49 outputs left out)

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-m3


Processing 28/30


INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-m3


Processing 29/30


In [7]:
from ragas import evaluate

def print_score(score):
    for key,val in score.items():
        print(f"{key}: {val}")

In [8]:
from datasets import Dataset

d = {
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths
}

final_dataset = Dataset.from_dict(d)

In [11]:
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall, answer_similarity, answer_correctness
from ragas.metrics.critique import harmfulness

from utils.llm import get_llm, get_embedding_model
from ragas.run_config import RunConfig


# Open Source
# llm = get_llm(model="llama3.1")
# embed_model = get_embedding_model(model="BAAI/bge-m3")

# Proprietary
llm = get_llm("gpt-4o-mini")
embed_model = get_embedding_model("text-embedding-3-small")

# Metrics
metrics = [faithfulness, answer_relevancy, context_precision, context_recall, answer_similarity, answer_correctness, harmfulness]

c:\Users\arkan\Downloads\University\TA\ta-rag-demo\env\lib\site-packages\langchain_core\_api\deprecation.py:139: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAIEmbeddings`.
  warn_deprecated(


In [12]:
first = final_dataset.select(range(0, 6))
first_score = evaluate(first, metrics=metrics, llm=llm, embeddings=embed_model)
print_score(first_score)

Evaluating:   0%|          | 0/42 [00:00<?, ?it/s]ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-26' coro=<as_completed.<locals>.sema_coro() done, defined at c:\Users\arkan\Downloads\University\TA\ta-rag-demo\env\lib\site-packages\ragas\executor.py:35> exception=AttributeError("'NoneType' object has no attribute 'embed_text'")>
Traceback (most recent call last):
  File "C:\Users\arkan\AppData\Local\Programs\Python\Python39\lib\asyncio\tasks.py", line 256, in __step
    result = coro.send(None)
  File "c:\Users\arkan\Downloads\University\TA\ta-rag-demo\env\lib\site-packages\ragas\executor.py", line 37, in sema_coro
    return await coro
  File "c:\Users\arkan\Downloads\University\TA\ta-rag-demo\env\lib\site-packages\ragas\executor.py", line 109, in wrapped_callable_async
    raise e
  File "c:\Users\arkan\Downloads\University\TA\ta-rag-demo\env\lib\site-packages\ragas\executor.py", line 104, in wrapped_callable_async
    result = await callable(*args,

faithfulness: 0.7132352941176471
answer_relevancy: 0.6041680568076401
context_precision: 0.9861111110733796
context_recall: 0.8333333333333334
answer_similarity: 0.8136261809777386
answer_correctness: 0.565915818125438
harmfulness: 0.0


In [13]:
second = final_dataset.select(range(6, 12))
second_score = evaluate(second, metrics=metrics, llm=llm, embeddings=embed_model)
print_score(second_score)

Evaluating:   0%|          | 0/42 [00:00<?, ?it/s]INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Evaluating:   2%|▏         | 1/42 [00:00<00:39,  1.04it/s]INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request:

faithfulness: 0.7747294372294373
answer_relevancy: 0.6210845402469612
context_precision: 0.8108465608231702
context_recall: 0.6944444444444445
answer_similarity: 0.7461735350489739
answer_correctness: 0.53824471388504
harmfulness: 0.16666666666666666


In [14]:
third = final_dataset.select(range(12, 18))
third_score = evaluate(third, metrics=metrics, llm=llm, embeddings=embed_model)
print_score(third_score)

Evaluating:   0%|          | 0/42 [00:00<?, ?it/s]INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Evaluating:   2%|▏         | 1/42 [00:01<00:57,  1.40s/it]INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Evaluating:   5%|▍         | 2/42 [00:01<00:29,  1.37it/s]INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PO

faithfulness: 0.7263888888888889
answer_relevancy: 0.8920599321997078
context_precision: 0.8977777777567667
context_recall: 0.5
answer_similarity: 0.7426714032031283
answer_correctness: 0.6992070724264523
harmfulness: 0.0


In [15]:
fourth = final_dataset.select(range(18, 24))
fourth_score = evaluate(fourth, metrics=metrics, llm=llm, embeddings=embed_model)
print_score(fourth_score)

Evaluating:   0%|          | 0/42 [00:00<?, ?it/s]INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 0.778000 seconds
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 0.858000 seconds
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 0.967000 seconds
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 0.772000 seconds
INFO:httpx:HTTP Request:

faithfulness: 0.6892195767195767
answer_relevancy: 0.69129785393086
context_precision: 0.8127777777415223
context_recall: 0.6666666666666666
answer_similarity: 0.6460081357736513
answer_correctness: 0.3848044287001073
harmfulness: 0.16666666666666666


In [17]:
fifth = final_dataset.select(range(24, 30))
fifth_score = evaluate(fifth, metrics=metrics, llm=llm, embeddings=embed_model)
print_score(fifth_score)

Evaluating:   0%|          | 0/42 [00:00<?, ?it/s]INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Evaluating:   2%|▏         | 1/42 [00:01<00:47,  1.16s/it]INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Evaluating:   5%|▍         | 2/42 [00:01<00:25,  1.58it/s]INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.co

faithfulness: 0.7255298078827491
answer_relevancy: 0.7839512210686667
context_precision: 0.6459242724627262
context_recall: 0.6666666666666666
answer_similarity: 0.6568027458162949
answer_correctness: 0.5200280066626765
harmfulness: 0.3333333333333333


In [18]:
import pandas as pd

score = pd.concat([first_score.to_pandas(), second_score.to_pandas(), third_score.to_pandas(), fourth_score.to_pandas(), fifth_score.to_pandas()])
score.describe()

,faithfulness,answer_relevancy,context_precision,context_recall,answer_similarity,answer_correctness,harmfulness
count,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000
mean,0.725821,0.718512,0.830687,0.672222,0.721056,0.541640,0.133333
std,0.247193,0.323682,0.298659,0.472386,0.191228,0.273587,0.345746
min,0.133333,0.000000,0.000000,0.000000,0.080916,0.020229,0.000000
25%,0.547980,0.652536,0.823053,0.000000,0.692434,0.343604,0.000000
50%,0.750000,0.840055,1.000000,1.000000,0.761444,0.511207,0.000000
75%,1.000000,0.955959,1.000000,1.000000,0.822362,0.783965,0.000000
max,1.000000,0.996211,1.000000,1.000000,0.928714,0.885218,1.000000


# Logs

```py
from utils.rag.basic_agent import caller_with_sources
```

|       | faithfulness | answer_relevancy | context_precision | context_recall | answer_similarity | answer_correctness | harmfulness |
|------:|-------------:|-----------------:|------------------:|---------------:|------------------:|-------------------:|------------:|
| count |    30.000000 |        30.000000 |         30.000000 |      30.000000 |         30.000000 |          30.000000 |   30.000000 |
|  mean |     0.714891 |         0.717273 |          0.762142 |       0.691667 |          0.629799 |           0.537138 |    0.300000 |
|   std |     0.366179 |         0.374074 |          0.339197 |       0.462766 |          0.377751 |           0.394079 |    0.466092 |
|   min |     0.000000 |         0.000000 |          0.000000 |       0.000000 |         -0.008620 |          -0.002155 |    0.000000 |
|   25% |     0.541667 |         0.740522 |          0.711982 |       0.000000 |          0.229678 |           0.156566 |    0.000000 |
|   50% |     0.845238 |         0.890021 |          0.928019 |       1.000000 |          0.798575 |           0.546770 |    0.000000 |
|   75% |     1.000000 |         0.959939 |          1.000000 |       1.000000 |          0.923533 |           0.928037 |    1.000000 |
|   max |     1.000000 |         1.000000 |          1.000000 |       1.000000 |          0.964361 |           0.991090 |    1.000000 |


```py
from utils.rag.open_source.basic_agent import caller_with_sources
```

|       | faithfulness | answer_relevancy | context_precision | context_recall | answer_similarity | answer_correctness | harmfulness |
|------:|-------------:|-----------------:|------------------:|---------------:|------------------:|-------------------:|------------:|
| count |    30.000000 |        30.000000 |         30.000000 |      30.000000 |         30.000000 |          30.000000 |   30.000000 |
|  mean |     0.725821 |         0.718512 |          0.830687 |       0.672222 |          0.721056 |           0.541640 |    0.133333 |
|   std |     0.247193 |         0.323682 |          0.298659 |       0.472386 |          0.191228 |           0.273587 |    0.345746 |
|   min |     0.133333 |         0.000000 |          0.000000 |       0.000000 |          0.080916 |           0.020229 |    0.000000 |
|   25% |     0.547980 |         0.652536 |          0.823053 |       0.000000 |          0.692434 |           0.343604 |    0.000000 |
|   50% |     0.750000 |         0.840055 |          1.000000 |       1.000000 |          0.761444 |           0.511207 |    0.000000 |
|   75% |     1.000000 |         0.955959 |          1.000000 |       1.000000 |          0.822362 |           0.783965 |    0.000000 |
|   max |     1.000000 |         0.996211 |          1.000000 |       1.000000 |          0.928714 |           0.885218 |    1.000000 |